# 사용자를 기억하는 에이전트 만들기

대부분의 에이전트는 대화를 매번 처음부터 시작합니다. 고객이 쇼핑 어시스턴트에 자기 사이즈, 예산, 피하는 소재를 알려 줬는데도 다음에 다시 찾아오면 에이전트는 전부 잊어버립니다. 고객은 같은 말을 반복해야 하고, 경험은 개인화된 것이 아니라 뻔한 것처럼 느껴집니다.

이 문제를 풀기 위해 Claude Managed Agents에 메모리를 도입했습니다. 에이전트가 고객마다 하나씩 갖는 공용 노트라고 생각하시면 됩니다. 세션 동안 Claude가 관련된 내용을 적어 두고, 같은 고객이 다시 찾아오면 그 메모가 그대로 남아 있습니다. 설정은 API 호출 한 번이면 됩니다.

이 가이드에서는 소매 브랜드를 위한 예시 쇼핑 어시스턴트를 만듭니다. 이 에이전트는 첫 방문 때 고객의 선호를 학습해 사용자별 메모리 저장소에 저장하고, 다음 방문에서는 따로 알려 주지 않아도 자동으로 그것을 떠올립니다.

## 만들 것

- 고객 한 명의 쇼핑 선호를 담는 **메모리 저장소**
- 그 저장소를 확인하고 갱신하도록 설정된 **쇼핑 에이전트**
- 방문을 넘어 메모리가 이어지는 것을 보여 주는 **별개의 두 세션**
- 여러분의 애플리케이션에서 메모리를 **조회하고 미리 채워 넣는** 패턴

## 메모리의 동작 방식

메모리 저장소를 세션에 붙이면 에이전트 환경 안의 `/mnt/memory/{store-name}` 디렉터리로 나타나고, Claude는 표준 파일 도구로 그곳의 파일을 읽고 씁니다. 여러분의 애플리케이션도 REST API로 같은 파일에 완전한 읽기·쓰기 권한을 갖습니다. 알고 있는 사실을 미리 채워 넣거나, 에이전트가 무엇을 썼는지 감사하거나, 전부 여러분의 시스템으로 내보낼 수 있습니다.

> **베타 기능.** 메모리 저장소는 Claude Managed Agents 퍼블릭 베타의 일부입니다. 정식 출시 전에 API가 바뀔 수 있습니다. Python SDK는 `client.beta` 아래의 모든 메서드에 필요한 `anthropic-beta` 헤더를 자동으로 추가합니다.

## 사전 준비

- Claude Managed Agents 베타에 접근할 수 있는 Anthropic API 키. `ANTHROPIC_API_KEY` 환경 변수로 설정하세요.
- Python 3.11 이상.
- Anthropic Python SDK. 메모리 저장소 메서드는 최신 릴리스가 필요합니다:

```bash
uv add anthropic
# or: pip install -U anthropic
```

## 클라이언트 설정

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0"

In [1]:
import os

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

### 대화 턴을 위한 헬퍼

매니지드 에이전트 세션은 이벤트 기반입니다. 사용자 메시지를 보낸 뒤 세션이 유휴 상태가 될 때까지 이벤트를 스트리밍합니다. 이 가이드의 나머지를 읽기 좋게 유지하기 위해, 그 루프를 에이전트의 답변과 `/mnt/memory/` 아래에서 건드린 파일을 출력하는 헬퍼로 감싸겠습니다.

In [2]:
def run_turn(session_id: str, user_text: str) -> str:
    """Send one user message and stream the agent's response until it goes idle.

    Returns the agent's full text reply for callers who want it programmatically.
    """
    print(f"\n[user]  {user_text}")
    reply_parts: list[str] = []

    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[
                {
                    "type": "user.message",
                    "content": [{"type": "text", "text": user_text}],
                }
            ],
        )
        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        reply_parts.append(block.text)
                        print(f"[agent] {block.text}")

            elif event.type == "agent.tool_use":
                # Surface reads and writes to the memory mount so you can see
                # the agent checking and updating its memory.
                inp = event.input or {}
                target = inp.get("file_path") or inp.get("command", "")
                if "/mnt/memory/" in str(target):
                    print(f"  [memory] {event.name}: {target}")

            elif event.type == "session.status_idle":
                # Break on any idle reason so an unexpected stop_reason
                # (such as requires_action) cannot hang the loop.
                break

            elif event.type == "session.status_terminated":
                break

    wait_for_idle_status(client, session_id)
    return "".join(reply_parts)

## 1단계: 메모리 저장소 만들기

메모리 저장소는 워크스페이스 범위의, 이름이 붙은 텍스트 파일 컨테이너입니다. 실제 배포에서는 보통 최종 사용자마다 저장소를 하나씩 만들고, 사용자 ID와 저장소 ID의 대응을 여러분의 데이터베이스에 보관합니다.

여기서 설정하는 `description`은 저장소가 붙을 때마다 에이전트의 시스템 프롬프트로 들어가므로, 이 저장소가 무엇을 위한 것인지 Claude에 알려 주는 데 사용하세요.

In [3]:
store = client.beta.memory_stores.create(
    name="Shopper Preferences",
    description=(
        "Personal shopping preferences for a single customer: "
        "sizes, style, budget, favorite brands, and materials to avoid."
    ),
)

print(store.id)  # memstore_01...

memstore_01NKkumXZXY8mEhoRA3xhBvN


## 2단계: 쇼핑 에이전트 정의하기

모든 매니지드 에이전트 세션에는 **에이전트**(모델, 시스템 프롬프트, 도구)와 **환경**(에이전트가 실행되는 컨테이너)이 필요합니다. 한 번 만들어 두면 여러 고객과 세션에서 재사용할 수 있습니다.

에이전트에 내장 `agent_toolset`을 주세요. 메모리를 읽고 쓰는 데 사용하는 파일 도구가 들어 있습니다. `agent_toolset_20260401`이라는 타입 문자열은 툴셋의 API 식별자이지 모델 별칭이 아니므로, 새 모델이 나와도 바꿀 필요가 없습니다.

In [4]:
environment = client.beta.environments.create(
    name="shopping-demo",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

agent = client.beta.agents.create(
    name="Personal Shopper",
    model=MODEL,
    system=(
        "You are a personal shopping assistant for a retail brand. "
        "Help the customer find products that match their taste and budget, "
        "and remember what you learn about them for future visits."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

print(agent.id)  # agent_01...

agent_011CaMbJoHMUyR5TjeFDtMG9


## 3단계: 첫 방문, 에이전트가 선호를 학습합니다

이제 세션을 시작하고 `resources` 배열로 메모리 저장소를 붙입니다. `instructions` 필드는 이번 세션에서 이 저장소를 어떻게 쓸지 Claude에 알려 주는, 연결별 안내입니다.

고객의 첫 방문이므로 저장소는 비어 있습니다. 출력의 `[memory]` 줄을 눈여겨보세요. 에이전트가 저장소를 확인하고, 아직 아무것도 없다는 것을 알고, 학습한 내용을 담은 새 파일을 씁니다.

In [5]:
memory_resource = {
    "type": "memory_store",
    "memory_store_id": store.id,
    "access": "read_write",
    "instructions": (
        "This customer's personal preferences: sizes, style, budget, and "
        "materials to avoid. Check it at the start of every conversation "
        "and update it whenever you learn something new."
    ),
}

session_one = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],
)

# The response tells you where the store is mounted inside the agent's environment.
for resource in session_one.resources:
    if resource.type == "memory_store":
        print(f"Mounted at {resource.mount_path}")

Mounted at /mnt/memory/shopper-preferences


In [6]:
run_turn(
    session_one.id,
    "Hi! I'm looking for a new jacket. A few things about me: I wear a size "
    "medium, I only buy vegan leather (no animal leather please), my budget "
    "is usually under $200, and I love earth tones. What would you suggest?",
)

[user]  Hi! I'm looking for a new jacket. A few things about me: I wear a size medium, I only buy vegan leather (no animal leather please), my budget is usually under $200, and I love earth tones. What would you suggest?
[agent] Let me check if I have any info about you on file, and save your preferences right away!
  [memory] bash: cat /mnt/memory/shopper-preferences 2>/dev/null || echo "No file found"
  [memory] write: /mnt/memory/shopper-preferences
  [memory] write: /mnt/memory/shopper-preferences/preferences.md
[agent] I've saved your preferences for future visits! Now, here are some great **vegan leather jacket picks in earth tones under $200** that I'd suggest:

---

### 🌿 Top Picks for You

#### 1. **Olive Moto Jacket** — ~$110–$150
A classic biker silhouette in a deep **olive green** faux leather. Look for styles with zip-detail sleeves and a fitted waist. Great for casual or edgy looks.
- *Why you'll love it:* Earthy tone, very on-trend, versatile for layering.

#### 2. **Cam

## 4단계: 에이전트가 저장한 것 살펴보기

에이전트가 마운트에 쓴 것은 모두 평범한 메모리 문서이며, 여러분의 애플리케이션이 API로 읽고, 수정하고, 삭제할 수 있습니다. 응답에 파일 내용까지 포함하려면 `view="full"`로 저장소 내용을 조회하세요.

이것이 제품에 "우리가 알고 있는 당신의 정보" 페이지를 만들거나, 메모리를 여러분의 데이터베이스에 동기화하거나, 에이전트가 잘못 적은 것을 사람 검토자가 고치게 하는 방법입니다.

In [7]:
page = client.beta.memory_stores.memories.list(
    store.id,
    view="full",
)

for memory in page.data:
    if memory.type == "memory":
        print(f"=== {memory.path} ===")
        print(memory.content)
        print()

=== /preferences.md ===
# Shopper Preferences

## Sizes
- Tops/Jackets: Medium

## Style
- Loves earth tones (browns, tans, olive, terracotta, camel, rust, etc.)
- Interested in jackets

## Budget
- Usually under $200

## Materials
- VEGAN LEATHER ONLY — absolutely no animal leather
- (No other material restrictions noted yet)

## Favorite Brands
- None noted yet

## Other Notes
- First visit; preferences collected 2026-04-23


고객이 언급한 사이즈, 예산, 소재, 색상 선호가 주제별로 정리된 파일(보통 `/preferences.md` 같은 이름)이 보일 것입니다. 파일 이름과 구조는 Claude가 알아서 정했습니다.

## 5단계: 재방문, 에이전트가 스스로 떠올립니다

여기가 핵심입니다. **완전히 새로운 세션**을 만들고 **같은 메모리 저장소**를 붙입니다. 고객은 자신의 선호를 하나도 다시 말하지 않지만, 에이전트는 메모리에서 그것을 읽어 추천을 맞춰 줍니다.

In [8]:
session_two = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],  # same store, new session
)

run_turn(
    session_two.id,
    "Hey, I'm back! I need a bag for work. Any recommendations?",
)

[user]  Hey, I'm back! I need a bag for work. Any recommendations?
[agent] Welcome back! Let me check your preferences before making any recommendations!
  [memory] read: /mnt/memory/shopper-preferences
  [memory] bash: ls /mnt/memory/shopper-preferences/
  [memory] read: /mnt/memory/shopper-preferences/preferences.md
[agent] Great news — I already have your preferences on file! Here's what I kept in mind for you:

- 🎨 **Style:** Earth tones (browns, tans, olive, camel, etc.)
- 💰 **Budget:** Under $200
- 🌱 **Materials:** Vegan leather only — no animal leather

With that in mind, here are some work bag recommendations that would suit you perfectly:

---

### 🛍️ Top Picks for a Work Bag

1. **Structured Tote in Camel Vegan Leather** — A polished, professional look in a warm earth tone. Fits a laptop + daily essentials. Great desk-to-commute bag.

2. **Olive Canvas & Vegan Leather Trim Laptop Bag** — A sleek crossbody/shoulder bag with a relaxed but put-together vibe. The canvas keeps it 

출력에서 에이전트가 답하기 전에 `/mnt/memory/shopper-preferences/`를 읽는 것을 눈여겨보세요. 그런 다음 비건 레더에 어스톤 계열이고 $200 미만인 가방을 추천합니다. 이번 메시지에는 그런 내용이 하나도 없었는데도요. 선호가 첫 세션에서 그대로 이어진 것입니다.

## 한 걸음 더: 프로덕션을 위한 패턴

### 기존 데이터로 저장소 미리 채우기

계정 프로필이나 구매 이력에서 고객에 대해 이미 알고 있는 것이 있다면, 첫 세션 전에 저장소에 써 두어 에이전트가 알고 시작하게 할 수 있습니다. 실제 애플리케이션에서는 세션을 만들기 전에 이 채우기 단계를 실행합니다. 여기서 데모 뒤에 나오는 것은 위의 학습-회상 흐름에 집중하기 위해서일 뿐입니다.

> **참고:** 노트북 끝의 정리 셀에서 `store`가 삭제되므로, 이 셀은 그 전에 실행하세요.

In [9]:
# Optional: run before the cleanup cell.
seeded = client.beta.memory_stores.memories.create(
    store.id,
    path="/purchase-history.md",
    content=(
        "## Recent purchases\n"
        "- Canvas tote, olive, $89 (Jan 2026)\n"
        "- Wool beanie, rust, $34 (Dec 2025)\n"
    ),
)
print(f"Seeded {seeded.path}")

Seeded /purchase-history.md


### 고객별 저장소와 공용 저장소 함께 쓰기

세션 하나에 최대 여덟 개의 메모리 저장소를 붙일 수 있고, 각각 접근 수준을 따로 지정합니다. 흔한 패턴은 고객마다 읽기·쓰기 저장소 하나에 더해, 모든 세션이 공유하는 브랜드 전역 지식 읽기 전용 저장소를 두는 것입니다.

```python
catalog = client.beta.memory_stores.create(
    name="Product Catalog Notes",
    description="Current promotions, sizing guidance, and stock notes.",
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[
        {
            "type": "memory_store",
            "memory_store_id": store.id,
            "access": "read_write",
            "instructions": "This customer's personal preferences.",
        },
        {
            "type": "memory_store",
            "memory_store_id": catalog.id,
            "access": "read_only",
            "instructions": "Brand-wide product guidance. Consult before recommending items.",
        },
    ],
)

# Remember to clean up the catalog store when you are done:
# client.beta.memory_stores.delete(catalog.id)
```

### 감사와 교정

메모리 저장소에 대한 모든 쓰기는 그 작업을 수행한 세션과 함께 변경 불가능한 버전으로 기록됩니다. 이력을 보려면 `client.beta.memory_stores.memory_versions.list(...)`를, 에이전트가 잘못 적은 파일을 고치려면 `client.beta.memory_stores.memories.update(...)`를 사용하세요. 전체 기능은 [메모리 API 레퍼런스](https://docs.anthropic.com/en/docs/managed-agents/memory)를 참고하세요.

## 정리

이 가이드를 따라 하며 만든 리소스를 삭제합니다.

In [10]:
wait_for_idle_status(client, session_one.id)
wait_for_idle_status(client, session_two.id)

client.beta.sessions.archive(session_one.id)
client.beta.sessions.archive(session_two.id)
client.beta.memory_stores.delete(store.id)
client.beta.agents.archive(agent.id)
client.beta.environments.archive(environment.id)

## 요약

다음 과정을 통해 방문을 넘어 고객을 기억하는 쇼핑 에이전트를 만들었습니다.

1. 고객을 위한 메모리 저장소 생성
2. `resources`로 각 세션에 저장소 연결
3. 새 선호를 학습할 때마다 Claude가 `/mnt/memory/` 아래 파일을 읽고 쓰게 하기
4. 메모리 API로 여러분의 애플리케이션에서 그 파일들을 조회하기

여기서부터는 여러분의 사용자 ID를 메모리 저장소 ID에 대응시키고, 기존 고객 데이터로 저장소를 미리 채우고, 브랜드 전역 지식을 담은 공용 읽기 전용 저장소를 위에 얹을 수 있습니다.

### 이 시리즈의 다른 노트북

- [`CMA_iterate_fix_failing_tests.ipynb`](CMA_iterate_fix_failing_tests.ipynb) — 출발점 노트북. 실패하는 테스트 스위트를 대상으로 실행-관찰-수정 루프를 돌며 에이전트, 환경, 세션, 파일 마운트, 스트리밍 이벤트 루프를 소개합니다.
- [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) — 프로덕션 구성 이야기. 볼트 기반 MCP 자격 증명, 오래 유지되는 연결 없이 사람 개입을 처리하는 `session.status_idled` 웹훅, 리소스 수명 주기 CRUD 동사를 다룹니다.

### 더 알아보기

- [Claude Managed Agents 개요](https://docs.anthropic.com/en/docs/managed-agents/overview)
- [메모리 저장소 API 레퍼런스](https://docs.anthropic.com/en/docs/managed-agents/memory)
- [세션 리소스](https://docs.anthropic.com/en/docs/managed-agents/memory#attach-a-memory-store-to-a-session)